# Test Atomic Agent Marketplace Deployment

This notebook tests the marketplace agent deployment functionality with the atomic agent.

In [ ]:
import mlrun

## Setup

1. Upload your `atomic-agent.tar.gz` source archive
2. Upload your `requirements.txt` file
3. Set the paths and secrets below

In [ ]:
# Configure paths (update these to match your setup)
SOURCE_TAR = "./atomic-agent.tar.gz"  # Path to source archive
REQUIREMENTS_FILE = "./requirements.txt"  # Path to requirements file
PROJECT_NAME = "agent-demo"

# Required secrets (update with your actual values)
OPENAI_BASE_URL = "https://openai.prod.ai-gateway.quantumblack.com/a192ddc9-2002-4302-9840-26ad5eb678da/v1"
OPENAI_API_KEY = "YOUR_JWT_TOKEN_HERE"  # Replace with actual token

## Create Agent Metadata

Define the agent metadata manually (in production, this would come from the marketplace backend)

In [ ]:
# Agent metadata (mimics what would come from marketplace backend)
agent_metadata = {
    "name": "atomic-agent",
    "version": "0.0.1",
    "author": "McKinsey",
    "description": "Atomic Agent powered by LangChain and A2A protocol",
    "kind": "atomic-agent",
    "protocol": "A2A",
    "framework": "LangChain",
    "asset_url": "",  # Will be set during deployment
    "requirements": [],  # Will load from file
    "default_base_image": "mlrun/mlrun",
    "default_port": 10000,
    "default_command": "sh",
    "default_args": [
        "-c",
        "PYTHONPATH=/home/mlrun_code/atomic_agent python -m atomic_agent"
    ],
    "inputs": [
        {"name": "AGENT_CONFIG_PATH", "type": "env", "required": False, "default": "/home/mlrun_code/atomic_agent/config/agent.yaml"},
        {"name": "REDIS_HOST", "type": "env", "required": False, "default": "redis-master.default-tenant.svc.cluster.local"},
        {"name": "REDIS_PASSWORD", "type": "env", "required": False, "default": "mlrun"},
        {"name": "AGENT_HOST", "type": "env", "required": False, "default": "nuclio-mlrun-x-atomic-agent-atomic-agent.default-tenant.svc.cluster.local"},
        {"name": "AGENT_PORT", "type": "env", "required": False, "default": "10000"},
        {"name": "UV_SYSTEM_PYTHON", "type": "env", "required": False, "default": "1"},
        {"name": "OPENAI_BASE_URL", "type": "env", "required": True},
        {"name": "OPENAI_API_KEY", "type": "secret", "required": True}
    ],
    "categories": ["agents", "ai", "nlp"],
    "default_workdir": "/home/mlrun_code/",
    "build_extra": ""
}

## Upload Source Archive to Project

This uploads the tar.gz to MLRun's artifact store so it can be used during deployment.

In [ ]:
# Get or create project
project = mlrun.get_or_create_project(PROJECT_NAME, context="./")

# Upload source archive as artifact
atomic_agent_artifact = project.log_artifact(
    "atomic-agent-source",
    local_path=SOURCE_TAR
)

source_url = atomic_agent_artifact.target_path
print(f"Source uploaded to: {source_url}")

## Import Agent and View Info

In [ ]:
# Import agent from marketplace
agent = mlrun.import_agent("atomic-agent", agent_metadata)

# Display agent information
agent.info()

## Deploy Agent

This will:
1. Build base image with requirements (slow, ~10 minutes)
2. Build image with source code
3. Deploy the application
4. Create API gateway
5. Store secrets securely (OPENAI_API_KEY as K8s secret, not plain env var)

In [ ]:
# Deploy the agent
url = agent.deploy(
    project=PROJECT_NAME,
    source=source_url,  # TODO: Will be auto-fetched from backend in future
    requirements=REQUIREMENTS_FILE,
    gateway_config={
        "name": "atomic-agent-gw",
        "authentication_mode": "none",
    },
    # Required inputs
    OPENAI_BASE_URL=OPENAI_BASE_URL,
    OPENAI_API_KEY=OPENAI_API_KEY,  # Stored securely as K8s secret
)

print(f"\n✅ Agent deployed successfully!")
print(f"URL: {url}")

## Test the Deployed Agent

In [ ]:
import requests
import urllib3
import uuid

# Disable SSL warnings for self-signed certs
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Construct full URL (add https:// if not present)
if not url.startswith("http"):
    test_url = f"https://{url}"
else:
    test_url = url

# A2A JSON-RPC request format
payload = {
    "jsonrpc": "2.0",
    "method": "message/send",
    "params": {
        "message": {
            "messageId": str(uuid.uuid4()),
            "role": "user",
            "parts": [
                {
                    "text": "Hello! Can you help me with a simple task?"
                }
            ]
        }
    },
    "id": 1
}

print(f"Testing agent at: {test_url}\n")

response = requests.post(test_url, json=payload, verify=False)

print(f"Status Code: {response.status_code}")

if response.status_code == 200:
    result = response.json()
    if "result" in result:
        print("\n✅ Agent Response:")
        artifacts = result["result"].get("artifacts", [])
        if artifacts:
            for artifact in artifacts:
                parts = artifact.get("parts", [])
                for part in parts:
                    if part.get("kind") == "text":
                        print(part.get("text"))
    else:
        print("\n❌ Error in response:")
        print(result)
else:
    print(f"\n❌ Request failed:")
    print(response.text)

## Test Redeployment (Should Be Fast!)

Redeploy with same agent instance - should reuse cached images.

In [ ]:
# Redeploy with same agent instance - should be much faster!
url2 = agent.deploy(
    project=PROJECT_NAME,
    source=source_url,
    requirements=REQUIREMENTS_FILE,
    gateway_config={
        "name": "atomic-agent-gw",
        "authentication_mode": "none",
    },
    OPENAI_BASE_URL=OPENAI_BASE_URL,
    OPENAI_API_KEY=OPENAI_API_KEY,
)

print(f"\n✅ Agent redeployed (should have been faster!)")
print(f"URL: {url2}")

## Test Convenience Function

Test the one-call `deploy_agent()` function.

**Note:** This won't work in current implementation since `import_agent` is called without `agent_metadata` parameter in `deploy_agent()`. This is a TODO for when backend is implemented.

In [ ]:
# This cell will fail until deploy_agent() is updated to pass agent_metadata
# Keeping for reference when backend is implemented

# url3 = mlrun.deploy_agent(
#     "atomic-agent",
#     project=PROJECT_NAME,
#     source=source_url,
#     requirements=REQUIREMENTS_FILE,
#     gateway_config={
#         "name": "atomic-agent-gw",
#         "authentication_mode": "none",
#     },
#     OPENAI_BASE_URL=OPENAI_BASE_URL,
#     OPENAI_API_KEY=OPENAI_API_KEY,
# )
# 
# print(f"\n✅ Agent deployed via convenience function!")
# print(f"URL: {url3}")

print("⚠️ Convenience function deploy_agent() not yet compatible with current implementation.")
print("   Waiting for marketplace backend to provide agent_metadata.")